# VibeCheck: CLIP-LoRA aesthetic classifier

We trained a LoRA-fine-tuned CLIP-ViT-B/32 image classifier on a labeled
photo set covering the 18 aesthetic categories under `data/eval/`. The
notebook walks through data loading, three baselines, our LoRA fine-tune,
a rank ablation, and the final comparison.

Systems compared on the same held-out test split:

1. Groq Vision (Llama-4 Scout) -- the production baseline.
2. CLIP zero-shot -- frozen CLIP with class-name prompts.
3. CLIP linear probe -- frozen CLIP features, sklearn logistic regression head.
4. CLIP + LoRA (ours) -- rank-8 adapters on the vision tower.

The full pipeline runs in roughly 15-25 minutes on a Colab T4 GPU.

To run on Colab: open this notebook directly from GitHub
(File &rarr; Open notebook &rarr; GitHub tab &rarr; `kaylynhl/vibecheck`,
branch `model-building`, file `notebooks/demo.ipynb`), set the runtime to
a T4 GPU, then click Run all. The first cell auto-clones the repo and
prompts you to upload `data_eval.zip` (a zip of `data/eval/` from a local
checkout). The Groq baseline cell loads predictions from
`data/eval/groq_predictions.json`, which is produced ahead of time by
`scripts/export_groq_baseline.py` (run locally with a `GROQ_API_KEY`); if
the file is missing the comparison row simply shows `-`.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp'}


def _has_real_data(eval_root: Path) -> bool:
    if not eval_root.exists():
        return False
    for cls_dir in eval_root.iterdir():
        if not cls_dir.is_dir() or cls_dir.name.startswith('.'):
            continue
        for f in cls_dir.iterdir():
            if f.is_file() and f.suffix.lower() in IMG_EXTS:
                return True
    return False


if IN_COLAB:
    if not Path('/content/vibecheck').exists():
        !git clone -q https://github.com/kaylynhl/vibecheck.git /content/vibecheck
        !cd /content/vibecheck && git checkout -q model-building && git pull -q --ff-only
    os.chdir('/content/vibecheck')
    print('cwd:', Path.cwd())

    # Install runtime deps Colab doesn't ship by default.
    !pip install -q peft pillow-heif >/dev/null 2>&1

    if not _has_real_data(Path('data/eval')):
        from google.colab import files
        print('Upload data_eval.zip (zip of data/eval/ from your Mac):')
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        # Wipe any leftover empty class folders (stale .gitkeep stubs) before unzipping.
        !rm -rf data/eval && unzip -q "{zip_name}" -d .
    n_classes = sum(
        1 for p in Path('data/eval').iterdir()
        if p.is_dir() and not p.name.startswith('.')
    )
    print(f'data/eval: {n_classes} class folders')
else:
    print('Local run: assumes you are inside the repo with data/eval/ populated.')

## 1. Setup

Install dependencies. On Colab the first run pulls down ~2 GB of weights;
subsequent runs are cached.

In [ ]:
# Colab-only: uncomment the next line on Colab. Locally, install in your venv.
# !pip install -q torch torchvision open_clip_torch transformers peft scikit-learn matplotlib seaborn pillow-heif tqdm

import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# HEIC support (iPhone photos)
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    print('pillow-heif not installed; HEIC files will be skipped')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {DEVICE}')

## 2. Data loading

Walks `data/eval/<aesthetic>/*.{jpg,png,heic}` and builds a flat list of
`(image_path, label)` pairs.

**Important**: this notebook expects the `data/eval/` directory to be
alongside the notebook (or one level up). On Colab, upload the directory
or mount Drive.

In [ ]:
# Resolve data root: works in repo (notebooks/ next to data/) and on Colab
# (where you upload data/eval/ to the working directory).
CANDIDATES = [Path('data/eval'), Path('../data/eval'), Path('/content/data/eval')]
DATA_ROOT = next((p for p in CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find data/eval/. Upload it to Colab or run from repo root.')
print(f'Data root: {DATA_ROOT.resolve()}')

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.heic', '.heif', '.webp'}

# Class list must match data/eval/<folder> names. Order = label index.
CLASSES = sorted(
    d.name for d in DATA_ROOT.iterdir()
    if d.is_dir() and not d.name.startswith('.')
)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'Found {len(CLASSES)} classes: {CLASSES}')

items: list[tuple[Path, int]] = []
for cls in CLASSES:
    cls_dir = DATA_ROOT / cls
    for p in cls_dir.iterdir():
        if p.suffix.lower() in IMG_EXTS:
            items.append((p, CLASS_TO_IDX[cls]))

print(f'Total photos: {len(items)}')
print('Per-class counts:')
for cls, count in Counter(CLASSES[lbl] for _, lbl in items).items():
    print(f'  {cls:30s} {count}')

### 2.1 Train / val / test split

Stratified 60 / 20 / 20 split so each class appears proportionally in every
fold. The smallest classes hold roughly 8 photos, which produces 1 to 2
test photos per rare class -- tight, but the headline metric is averaged
across the full set so the larger classes dominate the global accuracy.

In [ ]:
paths = np.array([str(p) for p, _ in items])
labels = np.array([lbl for _, lbl in items])

# 60/20/20 stratified split
X_temp, X_test, y_temp, y_test = train_test_split(
    paths, labels, test_size=0.2, stratify=labels, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED
)  # 0.25 of 0.8 = 0.2 overall

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## 3. Load CLIP

We use the HuggingFace `transformers` CLIP (`openai/clip-vit-base-patch32`)
because it integrates cleanly with `peft` for LoRA. Same weights as OpenAI's
released ViT-B/32 model.

Model size: ~88M params total, ~88M frozen. With LoRA rank=8 we'll add
~0.3M trainable params (~0.3% of the model).

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

n_params = sum(p.numel() for p in clip_model.parameters())
print(f'CLIP loaded: {n_params/1e6:.1f}M params total')

### 3.1 Image feature extraction (cached)

Encode every photo with CLIP once and cache the resulting embeddings. The
zero-shot baseline and the linear probe both reuse these features so we
only pay the encoder cost once.

In [ ]:
def load_image(path: str) -> Image.Image:
    return Image.open(path).convert('RGB')

@torch.no_grad()
def encode_images(paths: list[str], batch_size: int = 16) -> np.ndarray:
    feats = []
    for i in tqdm(range(0, len(paths), batch_size), desc='encoding'):
        batch = [load_image(p) for p in paths[i:i+batch_size]]
        inputs = clip_processor(images=batch, return_tensors='pt').to(DEVICE)
        f = clip_model.get_image_features(**inputs)
        f = F.normalize(f, dim=-1)
        feats.append(f.cpu().numpy())
    return np.concatenate(feats, axis=0)

feats_train = encode_images(X_train.tolist())
feats_val = encode_images(X_val.tolist())
feats_test = encode_images(X_test.tolist())
print(f'Feature dim: {feats_train.shape[1]}')

## 4. Baseline 1: CLIP zero-shot

We hand-write a small prompt template per class, encode the prompts with
CLIP's text tower, compute cosine similarity against the cached image
features, and take the argmax. No training. This is the floor that any
fine-tuned variant has to beat to justify itself.

In [ ]:
# Class prompts. Multiple templates per class, averaged, is a common
# zero-shot strength trick from the original CLIP paper.
PROMPT_TEMPLATES = [
    'a photo of a {} aesthetic room',
    'a photo in the {} aesthetic',
    'a {} style space',
    'an outfit in the {} aesthetic',
    'a {} vibe',
]

# Human-readable class names. Auto-derived from folder names; a few aesthetics
# get explicit overrides so the prompt reads more naturally to CLIP.
_PROMPT_NAME_OVERRIDES = {
    'mid_century_modern': 'mid-century modern',
    'lo_fi': 'lo-fi',
    'retro_70s': '70s retro',
    'cottagecore_dark': 'dark cottagecore',
    'e_girl': 'e-girl',
}
CLASS_PROMPT_NAMES = {
    cls: _PROMPT_NAME_OVERRIDES.get(cls, cls.replace('_', ' '))
    for cls in CLASSES
}

@torch.no_grad()
def class_prompt_features() -> np.ndarray:
    """Average the encoded prompt templates per class, l2-normalize."""
    per_class = []
    for cls in CLASSES:
        name = CLASS_PROMPT_NAMES[cls]
        prompts = [t.format(name) for t in PROMPT_TEMPLATES]
        inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(DEVICE)
        f = clip_model.get_text_features(**inputs)
        f = F.normalize(f, dim=-1).mean(dim=0)
        f = F.normalize(f, dim=-1)
        per_class.append(f.cpu().numpy())
    return np.stack(per_class, axis=0)

text_proto = class_prompt_features()  # (K, D)

def zero_shot_predict(feats: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (predicted_labels, full_logits) using cosine similarity."""
    sims = feats @ text_proto.T  # (N, K)
    return sims.argmax(axis=1), sims

zs_pred, zs_logits = zero_shot_predict(feats_test)
zs_acc = accuracy_score(y_test, zs_pred)
zs_top3 = top_k_accuracy_score(y_test, zs_logits, k=3, labels=list(range(len(CLASSES))))
print(f'CLIP zero-shot  top-1={zs_acc:.3f}  top-3={zs_top3:.3f}')

## 5. Baseline 2: CLIP linear probe

We fit a multinomial logistic regression on top of the frozen CLIP image
features. This is the standard measure of how well CLIP's pretrained
representations separate our 18 classes without any backbone fine-tuning.

With limited training data and 512-dimensional features, regularization is
the main lever, so we sweep the inverse-L2 strength `C` and pick the
setting with the highest validation accuracy.

In [ ]:
best_lp = None
for C in [0.01, 0.1, 1.0, 10.0]:
    lp = LogisticRegression(
        C=C, max_iter=2000, multi_class='multinomial', random_state=SEED
    )
    lp.fit(feats_train, y_train)
    val_acc = lp.score(feats_val, y_val)
    print(f'  C={C:>6}  val_acc={val_acc:.3f}')
    if best_lp is None or val_acc > best_lp[1]:
        best_lp = (lp, val_acc, C)

lp_model, _, lp_C = best_lp
print(f'\nBest C={lp_C}')
lp_pred = lp_model.predict(feats_test)
lp_logits = lp_model.predict_proba(feats_test)
lp_acc = accuracy_score(y_test, lp_pred)
lp_top3 = top_k_accuracy_score(y_test, lp_logits, k=3, labels=list(range(len(CLASSES))))
print(f'CLIP linear probe  top-1={lp_acc:.3f}  top-3={lp_top3:.3f}')

## 6. CLIP + LoRA fine-tune

We inject low-rank adapters into the CLIP vision encoder via `peft`, freeze
everything else, and train a single linear classification head together
with the LoRA parameters. Hyperparameters used below:

- LoRA rank `r=8`, alpha=16, dropout=0.1
- Target modules: `q_proj`, `k_proj`, `v_proj`, `out_proj` in every vision-transformer block
- Optimizer: AdamW, lr=1e-4 for LoRA params, lr=1e-3 for the head, weight decay=0.01
- Loss: cross-entropy with label smoothing 0.05
- 30 epochs with early stopping on validation accuracy (patience=6)
- Batch size 16
- Light augmentation: random resized crop, horizontal flip, mild color jitter

In [ ]:
from peft import LoraConfig, get_peft_model
from torchvision import transforms

# Image transforms: CLIP's own processor handles normalization, but for
# end-to-end training we want augmentation on the train split.
MEAN = clip_processor.image_processor.image_mean
STD = clip_processor.image_processor.image_std
IMG_SIZE = clip_processor.image_processor.crop_size['height']

train_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfm = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class VibeDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths = paths
        self.labels = labels
        self.tfm = tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = load_image(self.paths[i])
        return self.tfm(img), int(self.labels[i])

train_ds = VibeDataset(X_train.tolist(), y_train, train_tfm)
val_ds   = VibeDataset(X_val.tolist(),   y_val,   eval_tfm)
test_ds  = VibeDataset(X_test.tolist(),  y_test,  eval_tfm)

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

In [ ]:
class CLIPLoRAClassifier(nn.Module):
    """CLIP vision tower with LoRA adapters + a linear classification head."""
    def __init__(self, clip_model, num_classes: int, lora_r=8, lora_alpha=16, lora_dropout=0.1):
        super().__init__()
        # Wrap the vision tower in LoRA. peft injects rank-r adapters at the
        # listed target modules; the rest stays frozen.
        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
            lora_dropout=lora_dropout,
            bias='none',
        )
        self.vision = get_peft_model(clip_model.vision_model, lora_config)
        self.visual_projection = clip_model.visual_projection  # frozen
        for p in self.visual_projection.parameters():
            p.requires_grad = False
        self.classifier = nn.Linear(clip_model.config.projection_dim, num_classes)

    def forward(self, pixel_values):
        v = self.vision(pixel_values=pixel_values).pooler_output
        v = self.visual_projection(v)
        return self.classifier(v)

model = CLIPLoRAClassifier(clip_model, num_classes=len(CLASSES)).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.2f}M / {total/1e6:.1f}M  ({trainable/total*100:.2f}%)')

In [ ]:
@torch.no_grad()
def evaluate(loader) -> tuple[float, np.ndarray, np.ndarray]:
    model.eval()
    all_preds, all_logits, all_y = [], [], []
    for x, y in loader:
        x = x.to(DEVICE)
        logits = model(x).cpu().numpy()
        all_logits.append(logits)
        all_preds.append(logits.argmax(axis=1))
        all_y.append(y.numpy())
    preds = np.concatenate(all_preds)
    logits = np.concatenate(all_logits)
    y_true = np.concatenate(all_y)
    return accuracy_score(y_true, preds), preds, logits

# Different LRs for the LoRA params vs the classifier head
lora_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' in n]
optimizer = torch.optim.AdamW(
    [
        {'params': lora_params, 'lr': 1e-4, 'weight_decay': 0.01},
        {'params': head_params, 'lr': 1e-3, 'weight_decay': 0.01},
    ]
)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

EPOCHS = 30
PATIENCE = 6
best_val = 0.0
best_state = None
patience = 0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= len(train_ds)

    val_acc, _, _ = evaluate(val_loader)
    history['train_loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    print(f'epoch {epoch+1:>2}  loss={epoch_loss:.4f}  val_acc={val_acc:.3f}')

    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if v.requires_grad or True}
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1} (best val_acc={best_val:.3f})')
            break

if best_state is not None:
    model.load_state_dict(best_state, strict=False)
print(f'\nLoaded best checkpoint (val_acc={best_val:.3f})')

## 6.5 LoRA rank ablation

To assess how adapter capacity affects accuracy, we re-train the model at
six ranks: r in {1, 2, 4, 8, 16, 32}. Each run uses the same train/val/test
split and the same hyperparameters as the headline r=8 run above; only
the adapter rank changes. CLIP is reloaded from cache for each rank so
the LoRA injection starts from a clean base.

In [ ]:
import gc


def train_lora_with_rank(rank: int, *, epochs=30, patience=6, seed=42, verbose=False):
    """Train one CLIP-LoRA model at the given rank, return test metrics + per-class accuracy.

    Reloads CLIP from local cache for each call so peft adapters don't
    wrap-on-wrap across rank-sweep iterations. Adds ~5s per rank but keeps
    the function reentrant.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    fresh_clip = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE).eval()
    for p in fresh_clip.parameters():
        p.requires_grad = False

    m = CLIPLoRAClassifier(fresh_clip, num_classes=len(CLASSES), lora_r=rank).to(DEVICE)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)

    lora_params = [p for n, p in m.named_parameters() if p.requires_grad and 'classifier' not in n]
    head_params = [p for n, p in m.named_parameters() if p.requires_grad and 'classifier' in n]
    optimizer = torch.optim.AdamW(
        [
            {'params': lora_params, 'lr': 1e-4, 'weight_decay': 0.01},
            {'params': head_params, 'lr': 1e-3, 'weight_decay': 0.01},
        ]
    )
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

    best_val = 0.0
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        m.train()
        epoch_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = m(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x.size(0)
        epoch_loss /= len(train_ds)

        m.eval()
        with torch.no_grad():
            preds, ys = [], []
            for x, y in val_loader:
                preds.append(m(x.to(DEVICE)).argmax(1).cpu().numpy())
                ys.append(y.numpy())
            val_acc = accuracy_score(np.concatenate(ys), np.concatenate(preds))

        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        if verbose:
            print(f"  epoch {epoch+1:>2}  loss={epoch_loss:.4f}  val_acc={val_acc:.3f}")

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        m.load_state_dict(best_state, strict=False)

    m.eval()
    with torch.no_grad():
        preds, ys, logits_all = [], [], []
        for x, y in test_loader:
            logits = m(x.to(DEVICE)).cpu().numpy()
            logits_all.append(logits)
            preds.append(logits.argmax(1))
            ys.append(y.numpy())
        preds = np.concatenate(preds)
        ys = np.concatenate(ys)
        logits_all = np.concatenate(logits_all)

    top1 = accuracy_score(ys, preds)
    top3 = top_k_accuracy_score(ys, logits_all, k=3, labels=list(range(len(CLASSES))))
    per_class = np.full(len(CLASSES), np.nan)
    for i in range(len(CLASSES)):
        mask = ys == i
        if mask.sum() > 0:
            per_class[i] = (preds[mask] == i).mean()

    result = {
        'rank': rank,
        'trainable_params': trainable,
        'best_val_acc': best_val,
        'test_top1': top1,
        'test_top3': top3,
        'per_class_acc': per_class,
        'history': history,
    }
    del m, fresh_clip
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

In [ ]:
RANK_SWEEP = [1, 2, 4, 8, 16, 32]
rank_results = []
for r in RANK_SWEEP:
    print(f"=== training rank {r} ===")
    res = train_lora_with_rank(r, verbose=False)
    rank_results.append(res)
    print(f"  -> top-1={res['test_top1']:.3f}  top-3={res['test_top3']:.3f}  trainable={res['trainable_params']:,}")

In [ ]:
import matplotlib.ticker as mtick

ranks = [r['rank'] for r in rank_results]
top1s = [r['test_top1'] for r in rank_results]
top3s = [r['test_top3'] for r in rank_results]
params = [r['trainable_params'] for r in rank_results]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(ranks, top1s, 'o-', color='C0', linewidth=2, label='Top-1 accuracy')
ax1.plot(ranks, top3s, 's--', color='C0', alpha=0.5, label='Top-3 accuracy')
ax1.set_xscale('log', base=2)
ax1.set_xticks(ranks)
ax1.get_xaxis().set_major_formatter(mtick.ScalarFormatter())
ax1.set_xlabel('LoRA rank')
ax1.set_ylabel('Test accuracy', color='C0')
ax1.tick_params(axis='y', labelcolor='C0')
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='center right')

ax2 = ax1.twinx()
ax2.plot(ranks, params, '^:', color='C3', alpha=0.6, label='Trainable params')
ax2.set_ylabel('Trainable parameters', color='C3')
ax2.tick_params(axis='y', labelcolor='C3')
ax2.set_yscale('log')

plt.title('LoRA rank ablation: accuracy vs trainable parameters')
plt.tight_layout()
plt.show()

print("\nRank ablation summary:")
print(f"{'rank':>5}  {'top-1':>7}  {'top-3':>7}  {'params':>12}")
for r in rank_results:
    print(f"{r['rank']:>5}  {r['test_top1']:>7.3f}  {r['test_top3']:>7.3f}  {r['trainable_params']:>12,}")

### 6.5.1 Per-class accuracy by rank

The heatmap below shows test accuracy for each (rank, class) pair. Smaller
classes (those with roughly 10 photos) have only 1-3 test images, so their
per-cell accuracy is high-variance and should be read with that caveat in
mind. Larger classes (those capped at 40 photos) stabilize quickly and
are the more reliable signal.

In [ ]:
heat = np.array([r['per_class_acc'] for r in rank_results])

plt.figure(figsize=(max(10, 0.6 * len(CLASSES)), 4))
sns.heatmap(
    heat,
    annot=True,
    fmt='.2f',
    xticklabels=CLASSES,
    yticklabels=[f'r={r}' for r in RANK_SWEEP],
    cmap='YlOrBr',
    vmin=0,
    vmax=1,
    cbar_kws={'label': 'Test accuracy'},
)
plt.xticks(rotation=45, ha='right')
plt.title('Per-class test accuracy by LoRA rank')
plt.tight_layout()
plt.show()

In [ ]:
# Plot training curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history['train_loss'])
ax1.set_title('Train loss')
ax1.set_xlabel('epoch')
ax2.plot(history['val_acc'])
ax2.set_title('Val accuracy')
ax2.set_xlabel('epoch')
plt.tight_layout()
plt.show()

lora_acc, lora_pred, lora_logits = evaluate(test_loader)
lora_top3 = top_k_accuracy_score(
    np.concatenate([y.numpy() for _, y in test_loader]),
    lora_logits, k=3, labels=list(range(len(CLASSES))),
)
print(f'CLIP + LoRA  top-1={lora_acc:.3f}  top-3={lora_top3:.3f}')

## 7. Groq Vision baseline

Load the cached Groq predictions for the same test split (run
`scripts/export_groq_baseline.py` from the repo root first to populate
`data/eval/groq_predictions.json`).

If the file isn't there, this cell prints a warning and the comparison
skips Groq.

In [ ]:
groq_pred_path = DATA_ROOT / 'groq_predictions.json'
groq_pred = None
groq_acc = None

def groq_cache_key(path: str) -> str:
    """Match scripts/export_groq_baseline.py relative cache keys."""
    resolved = Path(path).resolve()
    try:
        return resolved.relative_to(DATA_ROOT.resolve()).as_posix()
    except ValueError:
        return str(resolved)

if groq_pred_path.exists():
    raw = json.loads(groq_pred_path.read_text())
    # raw is {relative_path_under_data_eval: predicted_class_name}. The
    # absolute-path fallback supports older caches generated before this
    # notebook was made Colab-portable.
    preds = []
    for p in X_test:
        key = groq_cache_key(p)
        pred_label = raw.get(key, raw.get(str(Path(p).resolve())))
        preds.append(CLASS_TO_IDX.get(pred_label, -1))
    groq_pred = np.array(preds)
    valid = groq_pred >= 0
    if valid.sum() == 0:
        print('Groq predictions cache found, but none matched the held-out test paths/classes.')
    else:
        groq_acc = accuracy_score(y_test[valid], groq_pred[valid])
        print(f'Groq Vision  top-1={groq_acc:.3f}  (on {valid.sum()}/{len(y_test)} test samples)')
else:
    print(f'No Groq predictions cache at {groq_pred_path}. Run scripts/export_groq_baseline.py first.')

## 8. Comparison table

The headline result for the report.

In [ ]:
import pandas as pd

rows = [
    {'system': 'CLIP zero-shot',     'top-1': zs_acc,   'top-3': zs_top3,   'trainable_params': 0},
    {'system': 'CLIP linear probe',  'top-1': lp_acc,   'top-3': lp_top3,   'trainable_params': lp_model.coef_.size},
    {'system': 'CLIP + LoRA (ours)', 'top-1': lora_acc, 'top-3': lora_top3, 'trainable_params': trainable},
]
if groq_acc is not None:
    rows.insert(0, {'system': 'Groq Vision (Llama-4 Scout)', 'top-1': groq_acc, 'top-3': None, 'trainable_params': None})

results = pd.DataFrame(rows)
results

### 8.1 Confusion matrices

Look at which classes get confused. If `scandinavian` and `japandi`
blur together, that's a *known* hard pair and worth discussing in the
report.

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='YlOrBr')
    plt.xticks(rotation=45, ha='right')
    plt.title(title)
    plt.ylabel('true')
    plt.xlabel('predicted')
    plt.tight_layout()
    plt.show()

plot_cm(y_test, zs_pred,   'CLIP zero-shot')
plot_cm(y_test, lp_pred,   'CLIP linear probe')
plot_cm(y_test, lora_pred, 'CLIP + LoRA (ours)')

### 8.2 Per-class accuracy

Where does LoRA help most? Where does it hurt?

In [ ]:
print('LoRA per-class report:')
print(classification_report(y_test, lora_pred, target_names=CLASSES, zero_division=0))

## 9. Save the model

Persist the trained LoRA adapters and classification head so they can be
loaded by the backend at inference time. The frozen CLIP backbone is not
saved -- it is re-downloaded from HuggingFace at load time.

In [ ]:
SAVE_DIR = Path('artifacts/clip_lora_vibe')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save LoRA adapters
model.vision.save_pretrained(SAVE_DIR / 'lora_adapters')
# Save classification head + class list
torch.save(model.classifier.state_dict(), SAVE_DIR / 'classifier.pt')
(SAVE_DIR / 'classes.json').write_text(json.dumps(CLASSES, indent=2))
print(f'Saved to {SAVE_DIR.resolve()}')

## 10. Notes for the writeup

The figures saved by the final cell below are sized for direct inclusion in
a short report. Beyond the headline comparison table and confusion
matrices, the rank-ablation plot and per-class heatmap support a
discussion of (a) why r=8 is a defensible production choice relative to
higher ranks, and (b) which classes the model handles confidently versus
those where small per-class test counts drive high variance.

A standard limitations section should note: the closed-set assumption
(any photo is mapped to one of the 18 trained classes; out-of-distribution
inputs are confidently mislabeled), the small per-class test counts in the
rare aesthetics, and the imbalance across classes that remains after
capping the larger ones at 40 samples.

## 11. Export figures for the report

Re-renders every figure produced above to PNG (300 DPI), bundles them
into a single zip, and triggers a download in Colab. Locally, the zip is
left at `plots_for_report.zip` in the working directory.

In [ ]:
import zipfile
from pathlib import Path

PLOTS_DIR = Path('plots_for_report')
PLOTS_DIR.mkdir(exist_ok=True)

def _save(fig, name):
    fig.savefig(PLOTS_DIR / f'{name}.png', dpi=300, bbox_inches='tight')
    plt.close(fig)


# 1. Headline LoRA r=8 training curve
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 4))
axA.plot(history['train_loss']); axA.set_title('Train loss (r=8)'); axA.set_xlabel('epoch')
axB.plot(history['val_acc']); axB.set_title('Val accuracy (r=8)'); axB.set_xlabel('epoch')
plt.tight_layout()
_save(fig, '01_lora_r8_training_curve')

# 2. Rank-ablation: dual-axis accuracy vs trainable params
ranks = [r['rank'] for r in rank_results]
top1s = [r['test_top1'] for r in rank_results]
top3s = [r['test_top3'] for r in rank_results]
params = [r['trainable_params'] for r in rank_results]
fig, axA = plt.subplots(figsize=(8, 5))
axA.plot(ranks, top1s, 'o-', color='C0', linewidth=2, label='Top-1 accuracy')
axA.plot(ranks, top3s, 's--', color='C0', alpha=0.5, label='Top-3 accuracy')
axA.set_xscale('log', base=2)
axA.set_xticks(ranks)
axA.get_xaxis().set_major_formatter(mtick.ScalarFormatter())
axA.set_xlabel('LoRA rank'); axA.set_ylabel('Test accuracy', color='C0')
axA.tick_params(axis='y', labelcolor='C0')
axA.set_ylim(0, 1); axA.grid(True, alpha=0.3); axA.legend(loc='center right')
axB = axA.twinx()
axB.plot(ranks, params, '^:', color='C3', alpha=0.6)
axB.set_ylabel('Trainable parameters', color='C3'); axB.tick_params(axis='y', labelcolor='C3')
axB.set_yscale('log')
plt.title('LoRA rank ablation: accuracy vs trainable parameters')
plt.tight_layout()
_save(fig, '02_rank_ablation_dualaxis')

# 3. Per-class accuracy heatmap by rank
heat = np.array([r['per_class_acc'] for r in rank_results])
fig = plt.figure(figsize=(max(10, 0.6 * len(CLASSES)), 4))
sns.heatmap(
    heat, annot=True, fmt='.2f',
    xticklabels=CLASSES, yticklabels=[f'r={r}' for r in RANK_SWEEP],
    cmap='YlOrBr', vmin=0, vmax=1, cbar_kws={'label': 'Test accuracy'},
)
plt.xticks(rotation=45, ha='right')
plt.title('Per-class test accuracy by LoRA rank')
plt.tight_layout()
_save(fig, '03_per_class_heatmap_by_rank')

# 4. Confusion matrices (zero-shot, linear probe, LoRA r=8)
def _cm_fig(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
    fig = plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='YlOrBr')
    plt.xticks(rotation=45, ha='right')
    plt.title(title); plt.ylabel('true'); plt.xlabel('predicted')
    plt.tight_layout()
    return fig

_save(_cm_fig(y_test, zs_pred,   'CLIP zero-shot'),     '04_cm_zero_shot')
_save(_cm_fig(y_test, lp_pred,   'CLIP linear probe'),  '05_cm_linear_probe')
_save(_cm_fig(y_test, lora_pred, 'CLIP + LoRA (r=8)'),  '06_cm_lora_r8')

# 5. Comparison table as a figure
fig, ax = plt.subplots(figsize=(9, 1 + 0.4 * len(results)))
ax.axis('off')
display_df = results.copy()
for col in ('top-1', 'top-3'):
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda v: f'{v:.3f}' if isinstance(v, float) else ('-' if v is None else v))
if 'trainable_params' in display_df.columns:
    display_df['trainable_params'] = display_df['trainable_params'].apply(
        lambda v: f'{v:,}' if isinstance(v, (int, float)) and v is not None else '-'
    )
table = ax.table(cellText=display_df.values, colLabels=display_df.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)
plt.title('Comparison: top-1 / top-3 accuracy and trainable parameter count')
_save(fig, '07_comparison_table')

# Bundle and (on Colab) download
zip_path = Path('plots_for_report.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(PLOTS_DIR.iterdir()):
        zf.write(p, arcname=p.name)

print(f'Saved {len(list(PLOTS_DIR.iterdir()))} figures to {PLOTS_DIR.resolve()}/')
print(f'Zipped to {zip_path.resolve()}')

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except ImportError:
    print('Local run: open the zip manually from the path above.')